In [1]:
import pandas as pd
!pip install Sastrawi
import Sastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, StopWordRemover, ArrayDictionary
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.3 MB/s eta 0:00:00


In [2]:
Raw_tweet = pd.read_csv("Raw_tweet.csv")
Raw_tweet.rename(columns={'tweetText': 'Tweet'}, inplace=True)
Raw_tweet['Sentimen']=""
display(Raw_tweet.head())

,id,Tweet,tweetURL,type,tweetAuthor,handle,geo,mentions,hashtags,replyCount,quoteCount,retweetCount,likeCount,views,bookmarkCount,createdAt,allMediaURL,videoURL,Sentimen
0,1878247059184365813,"Yuk, kita bikin tahun 2025 ini jadi milik kita...",https://x.com/psi_id/status/1878247059184365813,tweet,DPP PSI,@psi_id,Jakarta Capital Region,NaN,"#SemangatBaru,#Gaspol2025,#TahunBaru2025,#Sema...",32,0,0,7,327,0,1/12/2025 8:06,https://pbs.twimg.com/media/GhDfvB0aAAEIsoJ.jp...,NaN,
1,1878018603150385614,Mulai Tahun dengan Cerdas: Simak 5 Tips Resolu...,https://x.com/bartno/status/1878018603150385614,tweet,Ibnu Usmar,@bartno,Jakarta,NaN,"#TipsKeuangan,#Tabungan,#TahunBaru2025,#Resolu...",0,0,0,0,1,0,1/11/2025 16:58,NaN,NaN,
2,1878000424000901468,Kirim dipan dan meja rias Jakarta\nUntuk pemes...,https://x.com/BLORAJATI_1/status/1878000424000...,tweet,BLORA JATI PROJECT,@BLORAJATI_1,"Blora, Indonesia",NaN,"#dipan,#mejarias,#tahunbaru2025,#awaltahun",0,0,0,0,13,0,1/11/2025 15:46,https://pbs.twimg.com/ext_tw_video_thumb/18780...,https://video.twimg.com/ext_tw_video/187800013...,
3,1877987090409075119,Mari bersama-sama mewujudkan visi Rektor ITS d...,https://x.com/its_campus/status/18779870904090...,tweet,Institut Teknologi Sepuluh Nopember,@its_campus,"Kota Surabaya, Jawa Timur",NaN,"#ITSCampus,#ITSSurabaya,#ITSWorldClassUniversi...",0,0,0,1,172,0,1/11/2025 14:53,NaN,NaN,
4,1877703302994858207,"Terimakasih sebelumnya buat 2024, tahun itu ak...",https://x.com/sca_frisca/status/18777033029948...,tweet,Frisca A Maharti,@sca_frisca,Bandung,NaN,"#Bandung,#tahunbaru2025",0,0,0,0,6,0,1/10/2025 20:05,https://pbs.twimg.com/media/Gg7v7gjbAAAI4XQ.jpg,NaN,


##1. Data Understanding

In [8]:
#Cek tipe data
display(Raw_tweet.dtypes)

#Cek jumlah data
display(Raw_tweet.shape)

#Cek apakah ada baris kolom "Tweet" yang kosong
temp = (Raw_tweet["Tweet"].isnull().sum())
print(f"Jumlah baris kosong = {temp}")

#Cek apakah ada data yang terduplikasi
temp = (Raw_tweet.duplicated().sum())
print(f"Jumlah baris terduplikasi = {temp}")

,0
id,int64
Tweet,object
tweetURL,object
type,object
tweetAuthor,object
handle,object
geo,object
mentions,object
hashtags,object
replyCount,int64


(100, 19)

Jumlah baris kosong = 0
Jumlah baris terduplikasi = 0


##2. Text Prepocessing

In [4]:
#Hanya dibutuhkan kolom "Tweet" dan "Sentimen"
data = Raw_tweet[["Tweet","Sentimen"]]
display(data.head())

,Tweet,Sentimen
0,"Yuk, kita bikin tahun 2025 ini jadi milik kita...",
1,Mulai Tahun dengan Cerdas: Simak 5 Tips Resolu...,
2,Kirim dipan dan meja rias Jakarta\nUntuk pemes...,
3,Mari bersama-sama mewujudkan visi Rektor ITS d...,
4,"Terimakasih sebelumnya buat 2024, tahun itu ak...",


In [5]:
#Mengubah teks menjadi lower case
data.loc[:,'Tweet'] = data['Tweet'].str.lower()

#Normalisasi kata non-formal
norm = {'mw':'mau', 'dgn':'dengan', 'blm':'belum', 'bs':'bisa',
        '\n':'', ':':'', '#':' ', '?':'', ',':'',
        'banget':'sekali', 'tak':'tidak','meni':'sangat'}

def normalize(str_text):
  for i in norm:
    str_text = str_text.replace(i, norm[i])
  return str_text

data.loc[:,'Tweet'] = data['Tweet'].apply(normalize)

#Menghilangkan Stopword
stop_words = StopWordRemoverFactory().get_stop_words()
new_array = ArrayDictionary(stop_words)
stop_words_remover_new = StopWordRemover(new_array)

def Stopword(str_text):
  str_text = stop_words_remover_new.remove(str_text)
  return str_text

data.loc[:,'Tweet'] = data['Tweet'].apply(Stopword)

#Menghilangkan angka
data.replace('\d+', '', regex=True, inplace=True)

#Menghilangkan hyperlink karena banyak ditemukan pada dataset
def remove_link(text):
  return re.sub(r'http\S+', '', text)
data.loc[:,'Tweet'] = data['Tweet'].apply(remove_link)

#Menghilangkan emoticon
def remove_emoticons(text):
  emoticons_pattern = re.compile(u'([\U0001F600-\U0001F64F])|([\U0001F300-\U0001F5FF])|([\U0001F680-\U0001F6FF])|([\U0001F1E0-\U0001F1FF])', flags=re.UNICODE)
  return emoticons_pattern.sub(r'', text)
data.loc[:,'Tweet'] = data['Tweet'].apply(remove_emoticons)

display(data.head(10))
data.to_csv('Unstem_tweet.csv')

<ipython-input-5-faffcb71220e>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.replace('\d+', '', regex=True, inplace=True)


,Tweet,Sentimen
0,yuk bikin tahun jadi milik dalam mewujudkan i...,
1,mulai tahun cerdas simak tips resolusi keuang...,
2,kirim dipan meja rias jakartauntuk pemesanan h...,
3,bersama-sama mewujudkan visi rektor its membaw...,
4,terimakasih sebelumnya buat tahun aku banyak ...,
5,maklumaninfo perkhidmatan perpustidakaan umska...,
6,dukung asta cita presiden prabowopolsek tptm g...,
7,electrizendimulai langkah kecil nyalakan perub...,
8,menjenguk warga cibingbin blok wage rumahnya k...,
9,jalan-jalan ulurujangan lupa payung abu-abusel...,


In [6]:
#Tokenisasi, membagi kalimat menjadi token
tokenized = data['Tweet'].apply(lambda x: x.split())

#Stemming, mengubah menjadi kata dasar
def stemming(str_text):
  factory = StemmerFactory()
  stemmer = factory.create_stemmer()
  do = []
  for w in str_text:
    dt = stemmer.stem(w)
    do.append(dt)
  d_text = []
  d_text = ' '.join(do)
  print(d_text)
  return d_text

tokenized = tokenized.apply(stemming)

#Eksport tokens ke dalam file .csv untuk dilakukan sentimen analysis
tokenized.to_csv('Clean_tweet.csv')

yuk bikin tahun jadi milik dalam wujud impi sis n bro semangatbaru gaspol tahunbaru semangatsolidaritas solidaritastanpabatas indonesiamaju psi
mulai tahun cerdas simak tips resolusi uang tipskeuangan tabung tahunbaru resolusi
kirim dipan meja rias jakartauntuk mesan hubung kontidak bawah ya jl nasional blora-cepu watulumbung jiken kec jiken kab blora jawa tengah dipan mejarias tahunbaru awaltahun
sama wujud visi rektor its bawa kampus tuju masa depan lebih gemilang itscampus itssurabaya itsworldclassuniversity vivat pidatoawaltahun tahunbaru
terimakasih belum buat tahun aku banyak ajar lama aku isolasi alias hibernasi dan sekarang sambut penuh gembira semangat baru juang tuju baik masa yang baik tuhan  bandung tahunbaru
maklumaninfo khidmat perpustidakaan umskal tahun tahunbaru perpustidakaanumskal
dukung asta cita presiden prabowopolsek tptm gelar suluh anti narkoba smpn tanah putih tanjung lawan salampresisi polisiindonesia antinarkoba tahunbaru fyp jangkauanluas semuaorang sorot
el